# House Pricing — Regression Models for Real Estate Price Prediction

**Team:**
- Laura de Freitas Sales
- Luis Enrique Krulikowski

---

This project follows the **CRISP-DM** (Cross-Industry Standard Process for Data Mining)
methodology to build a Machine Learning solution for predicting residential property
sale prices.

## 1. Business Understanding

**Business context:** acting as data scientists for a digital platform that buys and
sells real estate (an "iBuyer"), the goal is to provide automated, accurate property
valuations. A reliable pricing model reduces appraisal time, lowers the risk of
mispricing inventory, and increases the company's competitiveness.

**Data mining goal:** build a regression model that predicts `price` from structural,
locational, and condition attributes of a property, with an error (MAE) low enough to
support pricing decisions in production.

**Success criteria:** minimize the Mean Absolute Error (MAE) in US dollars on a held-out
test set, while keeping R² at a level that demonstrates strong explanatory power.

**Dataset:** [House Pricing Dataset — Kaggle](https://www.kaggle.com/datasets/alyelbadry/house-pricing-dataset).
It contains 21,613 real estate transactions, with structural features (bedrooms,
bathrooms, living area, etc.), location (latitude, longitude, zip code) and condition
(quality grade, year built), where the target variable is the sale price (`price`).

## 2. Data Understanding

### 2.1 Environment Setup and Data Loading

In [ ]:
# Library imports (PEP 8 ordering: standard library, third-party, local)
import os
import warnings

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import seaborn as sns
from sklearn.model_selection import train_test_split

# Display / warning configuration
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

# Global constants
RANDOM_STATE = 42

In [ ]:
# Download the dataset from Kaggle and load it into a DataFrame
dataset_path = kagglehub.dataset_download("alyelbadry/house-pricing-dataset")
df_raw = pd.read_csv(os.path.join(dataset_path, "house_prices.csv"))

### 2.2 Data Quality Check

In [ ]:
# Data integrity: remove duplicate listings, keeping the most recent sale
print(f"Original shape: {df_raw.shape}")

# Convert to datetime to guarantee correct chronological ordering
df_raw["date"] = pd.to_datetime(df_raw["date"])
df_raw = df_raw.sort_values("date", ascending=True)
df_raw = df_raw.drop_duplicates(subset="id", keep="last")

print(f"\nShape after removing duplicate IDs: {df_raw.shape}\n")

# Initial overview of columns and data types
df_raw.info()

### 2.3 Exploratory Data Analysis (EDA)

In [ ]:
# Distribution analysis of the target variable (price)
plt.figure(figsize=(12, 5))

# Histogram with KDE
plt.subplot(1, 2, 1)
sns.histplot(df_raw["price"], kde=True, color="teal")
plt.title("Original Price Distribution", fontsize=12)
plt.xlabel("Price (US$)")

# Boxplot to spot outliers
plt.subplot(1, 2, 2)
sns.boxplot(x=df_raw["price"], color="teal")
plt.title("Price Boxplot (Outlier Detection)", fontsize=12)

plt.tight_layout()
plt.show()

# Insight: the long right tail justifies applying np.log1p later on.

In [ ]:
# Correlation matrix with a mask to hide the upper triangle (cleaner visual)
plt.figure(figsize=(14, 10))

# Only numeric columns are considered (clearer and avoids dtype errors)
correlation_matrix = df_raw.corr(numeric_only=True)

# Mask out the upper triangle, since the matrix is symmetric
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))

sns.heatmap(
    correlation_matrix,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    center=0,
    linewidths=0.5,
)
plt.title("Feature Correlation Matrix", fontsize=14)
plt.show()

# Insight: sqft_living and sqft_above are highly correlated (> 0.85).
# Keeping both would introduce multicollinearity.

In [ ]:
# Basic geospatial analysis
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df_raw,
    x="long",
    y="lat",
    hue="price",
    palette="viridis",
    alpha=0.5,
    size="price",
    sizes=(10, 200),
)
plt.title("Geographic Price Distribution", fontsize=12)
plt.legend(bbox_to_anchor=(1.05, 1), loc=2)
plt.show()

# Insight: there is a clear concentration of high prices in specific latitudes.

In [ ]:
# Interactive map of properties by price.
# lat / lon: latitude and longitude columns.
# color: colors each point according to the property price.
# size: also scales point size by price (pricier properties appear larger).
# hover_name: shows the property ID on hover.
# hover_data: extra tooltip fields (bedrooms, bathrooms, etc.).
# color_continuous_scale: color palette.
# mapbox_style: background map style ("open-street-map" gives a detailed basemap).
# zoom: initial zoom level.

fig = px.scatter_mapbox(
    df_raw,
    lat="lat",
    lon="long",
    color="price",
    size="price",
    hover_name="id",
    hover_data=["bedrooms", "bathrooms", "sqft_living", "grade"],
    color_continuous_scale=px.colors.diverging.RdYlGn,
    size_max=15,
    zoom=8,
    mapbox_style="open-street-map",
    title="Interactive Geographic Distribution of Properties by Price",
)

# Center the title and clean up the legend
fig.update_layout(
    title_text="Geographic Distribution of Properties by Price",
    title_x=0.5,
    legend_title_text="Price (US$)",
)

fig.show()

## 3. Data Preparation

### 3.1 Outlier Treatment (Business Rules and Integrity)

In [ ]:
df_processed = df_raw.copy()

# Remove properties with inconsistent data (0 bedrooms/bathrooms, or a
# data-entry error such as 33 bedrooms)
df_processed = df_processed[
    (df_processed["bedrooms"] > 0) & (df_processed["bedrooms"] < 11)
]
df_processed = df_processed[df_processed["bathrooms"] > 0]

# Area outlier treatment using the IQR rule, to focus on the standard market
q1_area = df_processed["sqft_living"].quantile(0.25)
q3_area = df_processed["sqft_living"].quantile(0.75)
iqr_area = q3_area - q1_area
upper_bound_area = q3_area + 1.5 * iqr_area

df_processed = df_processed[df_processed["sqft_living"] <= upper_bound_area]

### 3.2 Categorical / Ordinal Variable Mapping

In [ ]:
df_processed["waterfront"] = (
    df_processed["waterfront"].map({"N": 0, "Y": 1}).fillna(0)
)

condition_map = {"Poor": 1, "Fair": 2, "Average": 3, "Good": 4, "Very Good": 5}
df_processed["condition"] = df_processed["condition"].map(condition_map)

### 3.3 Feature Engineering

In [ ]:
df_processed["sale_year"] = df_processed["date"].dt.year
df_processed["house_age"] = df_processed["sale_year"] - df_processed["yr_built"]
df_processed["house_age"] = df_processed["house_age"].clip(lower=0)

df_processed["was_renovated"] = (
    df_processed["yr_renovated"].apply(lambda year: 1 if year > 0 else 0)
)
df_processed["has_basement"] = (
    df_processed["sqft_basement"].apply(lambda area: 1 if area > 0 else 0)
)

### 3.4 Removing Redundant Columns and Target Transformation

In [ ]:
# Drop IDs, raw dates, and areas that are highly collinear with sqft_living
# (based on the correlation heatmap above)
columns_to_drop = [
    "id", "date", "yr_built", "yr_renovated",
    "sqft_basement", "sqft_above", "sqft_living15", "sqft_lot15",
]
df_processed = df_processed.drop(columns=columns_to_drop)

# Log transform of the target (price) to reduce right-skew
df_processed["price_log"] = np.log1p(df_processed["price"])

print(f"Preprocessing complete. Remaining records: {len(df_processed)}")

### 3.5 Train / Validation / Test Split

In [ ]:
# X holds the features; y_log holds the log-transformed target
X = df_processed.drop(columns=["price", "price_log"])
y_log = df_processed["price_log"]

In [ ]:
# Split into 3 sets: 70% train, 15% validation, 15% test.
# First, carve out the final test set.
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y_log, test_size=0.15, random_state=RANDOM_STATE
)

# Then split the remainder into train and validation.
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.1765, random_state=RANDOM_STATE
)

### 3.6 Zip Code Target Encoding

In [ ]:
# Target-encode the zip code using only training data, to avoid leakage.
# We compute the median real price per zip code within the training set.
y_train_real = np.expm1(y_train)
zipcode_median_map = y_train_real.groupby(X_train["zipcode"]).median().to_dict()
global_median = y_train_real.median()

# Apply the encoding (median price rank) to all three sets
for dataset in (X_train, X_val, X_test):
    dataset["zipcode_rank"] = (
        dataset["zipcode"].map(zipcode_median_map).fillna(global_median)
    )
    dataset.drop(columns=["zipcode"], inplace=True)

print("Split and encoding finished.")
print(f"Train: {X_train.shape[0]} | Val: {X_val.shape[0]} | Test: {X_test.shape[0]}")

## 4. Modeling

### 4.1 Dependencies and Evaluation Utility

In [ ]:
%pip install catboost -q

In [ ]:
# Models and metrics
from sklearn.ensemble import (
    AdaBoostRegressor,
    BaggingRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
    RandomForestRegressor,
    StackingRegressor,
    VotingRegressor,
)
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor

from catboost import CatBoostRegressor  # !pip install catboost
from lightgbm import LGBMRegressor      # !pip install lightgbm
from xgboost import XGBRegressor        # !pip install xgboost

In [ ]:
# Global dictionary that stores the validation metrics of every model
model_results = {}


def evaluate_model(model_name, y_true_log, y_pred_log):
    """Convert log-scale predictions back to dollars and report metrics.

    Args:
        model_name: Label used in the printed report.
        y_true_log: Ground-truth target in log scale.
        y_pred_log: Predicted target in log scale.

    Returns:
        A dict with R2, MAE and RMSE computed on the real (dollar) scale.
    """
    y_true_real = np.expm1(y_true_log)
    y_pred_real = np.expm1(y_pred_log)

    mae = mean_absolute_error(y_true_real, y_pred_real)
    rmse = np.sqrt(mean_squared_error(y_true_real, y_pred_real))
    r2 = r2_score(y_true_real, y_pred_real)

    print(f"--- {model_name} ---")
    print(f"MAE:  US$ {mae:,.2f}")
    print(f"RMSE: US$ {rmse:,.2f}")
    print(f"R2:   {r2:.4f}\n")

    return {"R2": r2, "MAE": mae, "RMSE": rmse}

### 4.2 Linear Models

In [ ]:
# Pipeline definition (Scaler + Model)
pipe_lr = Pipeline([
    ("scaler", RobustScaler()),
    ("model", LinearRegression()),
])

# Parameter grid: a plain linear regression has few hyperparameters,
# so we only test fit_intercept.
param_grid_lr = {"model__fit_intercept": [True, False]}

# cv=5 ensures a robust cross-validation
grid_lr = GridSearchCV(
    pipe_lr, param_grid_lr, cv=5, scoring="neg_mean_absolute_error", n_jobs=-1
)
grid_lr.fit(X_train, y_train)

best_lr = grid_lr.best_estimator_
y_val_pred = best_lr.predict(X_val)

print(f"Best parameters found: {grid_lr.best_params_}")
model_results["Linear Regression"] = evaluate_model(
    "Linear Regression", y_val, y_val_pred
)

In [ ]:
# Pipeline definition (the scaler is applied correctly inside CV)
pipe_ridge = Pipeline([
    ("scaler", RobustScaler()),
    ("model", Ridge()),
])

# alpha controls the strength of the regularization (higher = simpler model)
param_grid_ridge = {
    "model__alpha": [0.01, 0.1, 1.0, 10.0, 100.0],
    "model__solver": ["auto", "cholesky", "sag"],
}

# neg_mean_absolute_error focuses on minimizing the error in US$
grid_ridge = GridSearchCV(
    pipe_ridge, param_grid_ridge, cv=5, scoring="neg_mean_absolute_error", n_jobs=-1
)
grid_ridge.fit(X_train, y_train)

best_ridge = grid_ridge.best_estimator_
y_val_pred = best_ridge.predict(X_val)

print(f"Best parameters found: {grid_ridge.best_params_}")
model_results["Ridge Regression"] = evaluate_model(
    "Ridge Regression", y_val, y_val_pred
)

In [ ]:
# Pipeline definition
pipe_lasso = Pipeline([
    ("scaler", RobustScaler()),
    ("model", Lasso(max_iter=10000)),  # higher max_iter to ensure convergence
])

# alpha is very sensitive in Lasso; small values tend to work best
param_grid_lasso = {"model__alpha": [0.0001, 0.001, 0.01, 0.1, 1.0]}

grid_lasso = GridSearchCV(
    pipe_lasso, param_grid_lasso, cv=5, scoring="neg_mean_absolute_error", n_jobs=-1
)
grid_lasso.fit(X_train, y_train)

best_lasso = grid_lasso.best_estimator_
y_val_pred = best_lasso.predict(X_val)

print(f"Best parameters found: {grid_lasso.best_params_}")
model_results["Lasso Regression"] = evaluate_model(
    "Lasso Regression", y_val, y_val_pred
)

In [ ]:
# Pipeline definition
pipe_en = Pipeline([
    ("scaler", RobustScaler()),
    ("model", ElasticNet(max_iter=10000)),
])

# alpha: overall penalty strength
# l1_ratio: 0 behaves like pure Ridge, 1 like pure Lasso
param_grid_en = {
    "model__alpha": [0.01, 0.1, 1.0, 10.0],
    "model__l1_ratio": [0.2, 0.5, 0.8],
}

grid_en = GridSearchCV(
    pipe_en, param_grid_en, cv=5, scoring="neg_mean_absolute_error", n_jobs=-1
)
grid_en.fit(X_train, y_train)

best_en = grid_en.best_estimator_
y_val_pred = best_en.predict(X_val)

print(f"Best parameters found: {grid_en.best_params_}")
model_results["Elastic Net"] = evaluate_model("Elastic Net", y_val, y_val_pred)

### 4.3 Instance-Based and Single-Tree Models

In [ ]:
# RobustScaler is CRITICAL here so distances are computed correctly
pipe_knn = Pipeline([
    ("scaler", RobustScaler()),
    ("model", KNeighborsRegressor()),
])

# n_neighbors: number of neighbors (K)
# weights: 'uniform' weighs all neighbors equally, 'distance' favors closer ones
param_grid_knn = {
    "model__n_neighbors": [3, 5, 7, 11, 15],
    "model__weights": ["uniform", "distance"],
    "model__metric": ["euclidean", "manhattan"],
}

grid_knn = GridSearchCV(
    pipe_knn, param_grid_knn, cv=5, scoring="neg_mean_absolute_error", n_jobs=-1
)
grid_knn.fit(X_train, y_train)

best_knn = grid_knn.best_estimator_
y_val_pred = best_knn.predict(X_val)

print(f"Best parameters found: {grid_knn.best_params_}")
model_results["K-NN Regressor"] = evaluate_model("K-NN Regressor", y_val, y_val_pred)

In [ ]:
# The scaler is kept for architectural consistency across the project
pipe_dt = Pipeline([
    ("scaler", RobustScaler()),
    ("model", DecisionTreeRegressor(random_state=RANDOM_STATE)),
])

# max_depth: maximum depth (avoids overfitting)
# min_samples_split: minimum samples required to create a new split
# min_samples_leaf: minimum samples required at a leaf node
param_grid_dt = {
    "model__max_depth": [None, 5, 10, 15, 20],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
}

grid_dt = GridSearchCV(
    pipe_dt, param_grid_dt, cv=5, scoring="neg_mean_absolute_error", n_jobs=-1
)
grid_dt.fit(X_train, y_train)

best_dt = grid_dt.best_estimator_
y_val_pred = best_dt.predict(X_val)

print(f"Best parameters found: {grid_dt.best_params_}")
model_results["Decision Tree"] = evaluate_model("Decision Tree", y_val, y_val_pred)

In [ ]:
pipe_svr = Pipeline([
    ("scaler", RobustScaler()),
    ("model", SVR()),
])

# C: regularization strength (penalty for errors)
# epsilon: width of the margin where errors are ignored
param_grid_svr = {
    "model__C": [0.1, 1, 10],
    "model__epsilon": [0.01, 0.1],
    "model__kernel": ["rbf"],  # RBF is the gold standard for non-linear data
}

# Note: SVR can be slow; n_jobs=-1 helps use all available cores.
grid_svr = GridSearchCV(
    pipe_svr, param_grid_svr, cv=5, scoring="neg_mean_absolute_error", n_jobs=-1
)
grid_svr.fit(X_train, y_train)

best_svr = grid_svr.best_estimator_
y_val_pred = best_svr.predict(X_val)

print(f"Best parameters found: {grid_svr.best_params_}")
model_results["SVR"] = evaluate_model("SVR", y_val, y_val_pred)

In [ ]:
pipe_mlp = Pipeline([
    ("scaler", RobustScaler()),
    ("model", MLPRegressor(random_state=RANDOM_STATE, max_iter=500)),
])

# hidden_layer_sizes: network architecture (e.g. one layer with 50 neurons)
# activation: activation function ('relu' is the most common choice)
# alpha: L2 regularization strength
param_grid_mlp = {
    "model__hidden_layer_sizes": [(50,), (100,)],
    "model__activation": ["relu", "tanh"],
    "model__alpha": [0.0001, 0.05],
}

grid_mlp = GridSearchCV(
    pipe_mlp, param_grid_mlp, cv=5, scoring="neg_mean_absolute_error", n_jobs=-1
)
grid_mlp.fit(X_train, y_train)

best_mlp = grid_mlp.best_estimator_
y_val_pred = best_mlp.predict(X_val)

print(f"Best parameters found: {grid_mlp.best_params_}")
model_results["MLP Regressor"] = evaluate_model("MLP Regressor", y_val, y_val_pred)

### 4.4 Bagging-Family Ensembles

In [ ]:
pipe_rf = Pipeline([
    ("scaler", RobustScaler()),
    ("model", RandomForestRegressor(random_state=RANDOM_STATE)),
])

# n_estimators: number of trees in the forest
# max_features: number of features considered at each split
# max_depth: maximum depth, to control tree growth
param_dist_rf = {
    "model__n_estimators": [100, 300, 500],
    "model__max_depth": [None, 10, 20, 30],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", "log2", None],
}

# n_iter=15 balances exploration against runtime
rf_rand = RandomizedSearchCV(
    pipe_rf,
    param_distributions=param_dist_rf,
    n_iter=15,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
rf_rand.fit(X_train, y_train)

best_rf = rf_rand.best_estimator_
y_val_pred = best_rf.predict(X_val)

print(f"Best parameters found: {rf_rand.best_params_}")
model_results["Random Forest"] = evaluate_model("Random Forest", y_val, y_val_pred)

In [ ]:
# Base estimator: a depth-limited tree, to keep individual learners simple
base_tree = DecisionTreeRegressor(max_depth=10, random_state=RANDOM_STATE)

# Note: BaggingRegressor does not strictly need scaling for trees, but the
# scaler is kept to follow the project's architectural pattern.
pipe_bagging = Pipeline([
    ("scaler", RobustScaler()),
    ("model", BaggingRegressor(estimator=base_tree, random_state=RANDOM_STATE)),
])

# n_estimators: how many simple trees the bagging ensemble will build
# max_samples: fraction of the dataset each tree sees (key for diversity)
param_dist_bagging = {
    "model__n_estimators": [10, 50, 100],
    "model__max_samples": [0.5, 0.7, 0.9, 1.0],
    "model__bootstrap": [True, False],
}

bagging_rand = RandomizedSearchCV(
    pipe_bagging,
    param_distributions=param_dist_bagging,
    n_iter=10,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
bagging_rand.fit(X_train, y_train)

best_bagging = bagging_rand.best_estimator_
y_val_pred_bagging = best_bagging.predict(X_val)

print(f"Best parameters found: {bagging_rand.best_params_}")
model_results["Bagging"] = evaluate_model("Bagging", y_val, y_val_pred_bagging)

In [ ]:
# Base estimator: a shallow tree (stump or slightly deeper) is ideal for AdaBoost
base_tree_boost = DecisionTreeRegressor(max_depth=5, random_state=RANDOM_STATE)

pipe_adaboost = Pipeline([
    ("scaler", RobustScaler()),
    ("model", AdaBoostRegressor(estimator=base_tree_boost, random_state=RANDOM_STATE)),
])

# n_estimators: number of models in the sequence
# learning_rate: contribution of each model to the final fit
# loss: error function to minimize ('linear', 'square' or 'exponential')
param_dist_adaboost = {
    "model__n_estimators": [50, 100, 200],
    "model__learning_rate": [0.01, 0.05, 0.1, 0.5],
    "model__loss": ["linear", "square"],
}

adaboost_rand = RandomizedSearchCV(
    pipe_adaboost,
    param_distributions=param_dist_adaboost,
    n_iter=10,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
adaboost_rand.fit(X_train, y_train)

best_adaboost = adaboost_rand.best_estimator_
y_val_pred_adaboost = best_adaboost.predict(X_val)

print(f"Best parameters found: {adaboost_rand.best_params_}")
model_results["AdaBoost"] = evaluate_model("AdaBoost", y_val, y_val_pred_adaboost)

### 4.5 Gradient Boosting Family

In [ ]:
pipe_gbr = Pipeline([
    ("scaler", RobustScaler()),
    ("model", GradientBoostingRegressor(random_state=RANDOM_STATE)),
])

# learning_rate: contribution of each new tree (gradient step size)
# n_estimators: number of boosting stages
# subsample: fraction of samples used to fit each base learner (aids generalization)
param_dist_gbr = {
    "model__n_estimators": [100, 500, 1000],
    "model__learning_rate": [0.01, 0.05, 0.1],
    "model__max_depth": [3, 4, 5],
    "model__subsample": [0.8, 0.9, 1.0],
    "model__min_samples_split": [2, 5, 10],
}

# n_iter=15 keeps runtime reasonable
gbr_rand = RandomizedSearchCV(
    pipe_gbr,
    param_distributions=param_dist_gbr,
    n_iter=15,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
gbr_rand.fit(X_train, y_train)

best_gbr = gbr_rand.best_estimator_
y_val_pred = best_gbr.predict(X_val)

print(f"Best parameters found: {gbr_rand.best_params_}")
model_results["Gradient Boosting"] = evaluate_model(
    "Gradient Boosting", y_val, y_val_pred
)

In [ ]:
pipe_xgb = Pipeline([
    ("scaler", RobustScaler()),
    ("model", XGBRegressor(random_state=RANDOM_STATE, verbosity=0)),
])

# n_estimators: number of trees
# learning_rate: step size shrinkage
# max_depth: maximum tree depth
# colsample_bytree: fraction of columns sampled per tree
# gamma: minimum loss reduction required to make a split (regularization)
param_dist_xgb = {
    "model__n_estimators": [100, 500, 1000],
    "model__learning_rate": [0.01, 0.05, 0.1, 0.2],
    "model__max_depth": [3, 5, 7, 9],
    "model__colsample_bytree": [0.7, 0.8, 0.9],
    "model__subsample": [0.7, 0.8, 0.9],
    "model__gamma": [0, 0.1, 0.2],
}

# n_iter=15 balances exploration with CPU time
xgb_rand = RandomizedSearchCV(
    pipe_xgb,
    param_distributions=param_dist_xgb,
    n_iter=15,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
xgb_rand.fit(X_train, y_train)

best_xgb = xgb_rand.best_estimator_
y_val_pred = best_xgb.predict(X_val)

print(f"Best parameters found: {xgb_rand.best_params_}")
model_results["XGBoost"] = evaluate_model("XGBoost", y_val, y_val_pred)

In [ ]:
pipe_lgbm = Pipeline([
    ("scaler", RobustScaler()),
    ("model", LGBMRegressor(random_state=RANDOM_STATE, verbosity=-1)),
])

# num_leaves: main parameter controlling tree complexity
# learning_rate: step size
# n_estimators: number of boosting iterations
# min_child_samples: minimum samples per leaf (avoids overfitting)
# subsample: fraction of columns/rows used per iteration
param_dist_lgbm = {
    "model__n_estimators": [100, 500, 1000],
    "model__learning_rate": [0.01, 0.05, 0.1],
    "model__num_leaves": [31, 50, 70, 100],
    "model__max_depth": [-1, 10, 20],  # -1 means no limit
    "model__min_child_samples": [20, 30, 50],
    "model__subsample": [0.7, 0.8, 0.9],
}

lgbm_rand = RandomizedSearchCV(
    pipe_lgbm,
    param_distributions=param_dist_lgbm,
    n_iter=15,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
lgbm_rand.fit(X_train, y_train)

best_lgbm = lgbm_rand.best_estimator_
y_val_pred = best_lgbm.predict(X_val)

print(f"Best parameters found: {lgbm_rand.best_params_}")
model_results["LightGBM"] = evaluate_model("LightGBM", y_val, y_val_pred)

In [ ]:
# CatBoost does not require scaling, but RobustScaler is kept for
# project-wide consistency.
pipe_cat = Pipeline([
    ("scaler", RobustScaler()),
    ("model", CatBoostRegressor(
        random_state=RANDOM_STATE, verbose=0, allow_writing_files=False
    )),
])

# iterations: equivalent to n_estimators (number of trees)
# learning_rate: step size
# depth: tree depth (CatBoost uses symmetric trees)
# l2_leaf_reg: L2 regularization term
param_dist_cat = {
    "model__iterations": [100, 500, 1000],
    "model__learning_rate": [0.01, 0.05, 0.1],
    "model__depth": [4, 6, 8, 10],
    "model__l2_leaf_reg": [1, 3, 5, 7, 9],
}

# CatBoost is slower, so 10 iterations are enough for a good fit
cat_rand = RandomizedSearchCV(
    pipe_cat,
    param_distributions=param_dist_cat,
    n_iter=10,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
cat_rand.fit(X_train, y_train)

best_cat = cat_rand.best_estimator_
y_val_pred = best_cat.predict(X_val)

print(f"Best parameters found: {cat_rand.best_params_}")
model_results["CatBoost"] = evaluate_model("CatBoost", y_val, y_val_pred)

In [ ]:
pipe_hgb = Pipeline([
    ("scaler", RobustScaler()),
    ("model", HistGradientBoostingRegressor(random_state=RANDOM_STATE)),
])

# max_iter: number of boosting iterations (trees)
# max_leaf_nodes: controls the complexity of each tree
param_dist_hgb = {
    "model__max_iter": [100, 200, 300],
    "model__learning_rate": [0.01, 0.05, 0.1],
    "model__max_leaf_nodes": [31, 50, 100],
    "model__min_samples_leaf": [20, 50, 100],
}

hgb_rand = RandomizedSearchCV(
    pipe_hgb,
    param_distributions=param_dist_hgb,
    n_iter=10,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
hgb_rand.fit(X_train, y_train)

best_hgb = hgb_rand.best_estimator_
model_results["HistGradientBoosting"] = evaluate_model(
    "HistGradientBoosting", y_val, best_hgb.predict(X_val)
)

### 4.6 Combination Strategies: Voting, Blending, Stacking

In [ ]:
# Committee built from the 5 simple models tuned previously:
# Ridge, Lasso, ElasticNet, KNN and Decision Tree
voting = VotingRegressor(
    estimators=[
        ("ridge", best_ridge),
        ("lasso", best_lasso),
        ("elastic", best_en),
        ("knn", best_knn),
        ("dt", best_dt),
    ]
)

# Each base model is already a Pipeline, so Voting respects each one's scaler
voting.fit(X_train, y_train)

y_val_pred_voting = voting.predict(X_val)
model_results["Voting"] = evaluate_model("Voting", y_val, y_val_pred_voting)

In [ ]:
# Generate the predictions ("opinions") of each model on the VALIDATION set,
# reusing the models already fitted above.
val_pred_ridge = best_ridge.predict(X_val)
val_pred_lasso = best_lasso.predict(X_val)
val_pred_elastic = best_en.predict(X_val)
val_pred_knn = best_knn.predict(X_val)
val_pred_dt = best_dt.predict(X_val)

# Build the meta-dataset: each column is one base model's prediction
X_val_meta = np.column_stack((
    val_pred_ridge,
    val_pred_lasso,
    val_pred_elastic,
    val_pred_knn,
    val_pred_dt,
))

# Train the meta-model (the "judge") that learns the optimal weight per model
meta_model = LinearRegression()
meta_model.fit(X_val_meta, y_val)

y_val_pred_blending = meta_model.predict(X_val_meta)

base_model_names = ["Ridge", "Lasso", "ElasticNet", "KNN", "Decision Tree"]
print("Weights assigned by Blending:")
for name, weight in zip(base_model_names, meta_model.coef_):
    print(f"- {name}: {weight:.4f}")
print()

model_results["Blending"] = evaluate_model("Blending", y_val, y_val_pred_blending)

In [ ]:
# Base estimators: the simple models already tuned above
stacking_estimators = [
    ("ridge", best_ridge),
    ("lasso", best_lasso),
    ("elastic", best_en),
    ("knn", best_knn),
    ("dt", best_dt),
]

# Meta-model (final estimator): Ridge keeps the final decision regularized
final_meta_model = Ridge(alpha=1.0)

# cv=5 ensures the meta-model's training predictions are generated fairly
# (out-of-fold), avoiding leakage from the base models.
stacking_model = StackingRegressor(
    estimators=stacking_estimators,
    final_estimator=final_meta_model,
    cv=5,
    n_jobs=-1,
)
stacking_model.fit(X_train, y_train)

y_val_pred_stacking = stacking_model.predict(X_val)
model_results["Stacking"] = evaluate_model("Stacking", y_val, y_val_pred_stacking)

In [ ]:
# Nested Stacking: a two-level ensemble that combines a "simple models" stack
# with a "boosting models" stack, then blends both with a final meta-model.
#
# NOTE (fix applied during refactor): the original notebook tried to reload
# these models from disk with joblib.load(...), but never saved them in a
# prior cell, which would crash on a fresh run. Here we reuse the estimators
# that are already fitted and held in memory.
print("Assembling the nested stacking ensemble...")

stack_simple = StackingRegressor(
    estimators=[
        ("ridge", best_ridge),
        ("lasso", best_lasso),
        ("knn", best_knn),
        ("dt", best_dt),
    ],
    final_estimator=LinearRegression(),
    cv=5,
)

stack_boosting = StackingRegressor(
    estimators=[
        ("cat", best_cat),
        ("lgbm", best_lgbm),
        ("xgb", best_xgb),
    ],
    final_estimator=LinearRegression(),
    cv=5,
)

# Level-2 "master stack": combines the predictions of both sub-stacks
print("Training the nested ensemble (this may take a few minutes)...")
nested_stack = StackingRegressor(
    estimators=[
        ("stack_simple", stack_simple),
        ("stack_boosting", stack_boosting),
    ],
    final_estimator=Ridge(alpha=1.0),  # Ridge at the top avoids overfitting
    cv=5,
    n_jobs=-1,
)
nested_stack.fit(X_train, y_train)

y_val_pred_nested = nested_stack.predict(X_val)
model_results["Nested Stacking"] = evaluate_model(
    "Nested Stacking", y_val, y_val_pred_nested
)

## 5. Evaluation

### 5.1 Validation Results Comparison

In [ ]:
results_df = pd.DataFrame(model_results).T.sort_values("MAE")
results_df

### 5.2 Final Model Selection and Test Set Evaluation

> **Addition made during the refactor.** The original notebook compared every
> model on the validation set but never confirmed performance on the
> held-out test set, which is the actual CRISP-DM Evaluation step (estimating
> how the model will generalize to unseen data). The cell below selects the
> model with the lowest validation MAE and evaluates it once on the test set.

In [ ]:
# Map each model name to a callable that returns log-scale predictions,
# so every model (including the two-stage Blending ensemble) can be scored
# the same way.
predict_fns = {
    "Linear Regression": best_lr.predict,
    "Ridge Regression": best_ridge.predict,
    "Lasso Regression": best_lasso.predict,
    "Elastic Net": best_en.predict,
    "K-NN Regressor": best_knn.predict,
    "Decision Tree": best_dt.predict,
    "SVR": best_svr.predict,
    "MLP Regressor": best_mlp.predict,
    "Random Forest": best_rf.predict,
    "Gradient Boosting": best_gbr.predict,
    "XGBoost": best_xgb.predict,
    "LightGBM": best_lgbm.predict,
    "CatBoost": best_cat.predict,
    "HistGradientBoosting": best_hgb.predict,
    "Voting": voting.predict,
    "Stacking": stacking_model.predict,
    "Nested Stacking": nested_stack.predict,
    "Blending": lambda X: meta_model.predict(
        np.column_stack((
            best_ridge.predict(X),
            best_lasso.predict(X),
            best_en.predict(X),
            best_knn.predict(X),
            best_dt.predict(X),
        ))
    ),
}

best_model_name = results_df["MAE"].idxmin()
print(f"Best model on validation (lowest MAE): {best_model_name}\n")

y_test_pred = predict_fns[best_model_name](X_test)
test_metrics = evaluate_model(f"{best_model_name} (Test Set)", y_test, y_test_pred)

## 6. Deployment

For this academic project, "deployment" is limited to persisting the winning
model artifact with `joblib`, so it can later be loaded by a scoring script
or an API endpoint. In a production setting, this phase would also cover
monitoring for data/concept drift, a retraining schedule, and a rollback plan.

In [ ]:
import joblib

MODEL_OUTPUT_PATH = "best_house_pricing_model.joblib"

# Estimators (e.g. Voting, Stacking, Nested Stacking) can be pickled directly.
# Blending is a custom function, not a single estimator, so we persist its
# two components instead.
if best_model_name == "Blending":
    joblib.dump(
        {
            "base_models": {
                "ridge": best_ridge,
                "lasso": best_lasso,
                "elastic": best_en,
                "knn": best_knn,
                "dt": best_dt,
            },
            "meta_model": meta_model,
        },
        MODEL_OUTPUT_PATH,
    )
else:
    joblib.dump(predict_fns[best_model_name].__self__, MODEL_OUTPUT_PATH)

print(f"Model '{best_model_name}' saved to: {MODEL_OUTPUT_PATH}")